# 4. ICA as a spatial filter (rough sketch)

**Idea.** ICA splits each participant's EEG into components. Each component is a fixed weighting of the electrodes, i.e. a **spatial filter**: its time course is one clean "virtual channel". So far ICA was used to *remove* artifact components. Here it is used to *keep* the brain components we care about:

- an **alpha component**: strongest over parietal/occipital electrodes, with a peak at 8–12 Hz;
- a **beta component**: strongest over central (sensorimotor) electrodes, with power at 13–30 Hz.

Each participant's chosen components replace the ROI averages: their alpha and beta **amplitude envelopes** are the input for cRQA.

**Why:** an electrode records a mixture of sources (and the mastoid reference). A component isolates one source, so alpha and beta are cleaner and less mixed with each other, with eye/muscle activity, or with the reference. In the pilot, alpha was not visible at the electrode level; a component can reveal it.

**Steps:** 1. load clean data + ICA → 2. score every kept component (alpha / beta spectrum × location) → 3. pick the best per band, check by eye → 4. envelopes of the chosen components → 5. save.

*Rough sketch: the scoring rule and channel groups are first guesses, to discuss.*

In [ ]:
import mne
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import config as cfg
%matplotlib inline

## 0. Participant and settings

In [ ]:
import os

SUBJECT = os.environ.get("RT_SUBJECT", "sub-001")
OUT_DIR = cfg.out_dir(SUBJECT)

BANDS = {"alpha": (8, 12), "beta": (13, 30)}
BACKGROUND = [(2, 7), (31, 40)]  # used to fit the 1/f background
MIN_LOCATION = 0.4  # share of the component's weight in the region
REGIONS = {  # where each component should be strongest
    "alpha": ["P3", "Pz", "P4", "P7", "P8", "O1", "O2", "CP1", "CP2"],
    "beta": ["C3", "Cz", "C4", "FC1", "FC2", "CP1", "CP2"],
}

## 1. Load the clean data and the ICA

In [ ]:
raw = mne.io.read_raw(OUT_DIR / f"{SUBJECT}_clean_raw.fif", preload=True)
ica = mne.preprocessing.read_ica(OUT_DIR / f"{SUBJECT}_ica.fif")
removed = pd.read_csv(OUT_DIR / f"{SUBJECT}_badcomps.csv")["component"].astype(int).tolist()
kept = [c for c in range(ica.n_components_) if c not in removed]

sources = ica.get_sources(raw)
print(f"{len(kept)} kept components; removed as artifacts: {removed}")

## 2. Score every kept component

- **Peak (dB):** EEG power falls with frequency (the "1/f" background). A straight line is fitted to the spectrum (log–log) outside the bands; the peak is how far the band rises above that line (> 0 dB = a real peak).
- **Location:** share of the component's weight on the band's region.
- **Choice:** the highest peak among components with at least 40% of their weight in the region. Only task data (not the breaks) is used.

In [ ]:
psd = sources.compute_psd(picks=[f"ICA{c:03d}" for c in kept], fmin=2, fmax=40, reject_by_annotation=True)
freqs, power = psd.freqs, psd.get_data()

# 1/f background: straight line in log-log, fitted outside the bands
log_f, log_p = np.log10(freqs), 10 * np.log10(power)
fit = np.zeros(len(freqs), bool)
for lo, hi in BACKGROUND:
    fit |= (freqs >= lo) & (freqs <= hi)
slope, offset = np.polyfit(log_f[fit], log_p[:, fit].T, 1)
above = log_p - (slope[:, None] * log_f + offset[:, None])  # dB above the background

maps = np.abs(ica.get_components())  # (channels, components)
ch = ica.info["ch_names"]

scores = pd.DataFrame({"component": kept})
for band, (lo, hi) in BANDS.items():
    region = [ch.index(c) for c in REGIONS[band] if c in ch]
    scores[f"{band}_peak_dB"] = above[:, (freqs >= lo) & (freqs <= hi)].mean(axis=1)
    scores[f"{band}_location"] = maps[region][:, kept].sum(axis=0) / maps[:, kept].sum(axis=0)
scores.round(2).sort_values("alpha_peak_dB", ascending=False)

## 3. Pick the best component per band, and check it

The best-scoring component is a suggestion: look at its map and spectrum. Overrule it in `CHOSEN` if needed (e.g. a muscle component can score high for beta).

In [ ]:
CHOSEN = {}
for band in BANDS:
    candidates = scores[scores[f"{band}_location"] >= MIN_LOCATION]
    if len(candidates) == 0:
        print(f"{band}: no component is mostly in the region; choose by hand")
        continue
    best = candidates.loc[candidates[f"{band}_peak_dB"].idxmax()]
    CHOSEN[band] = int(best["component"])
    print(f"{band}: ICA{CHOSEN[band]:03d}, peak {best[f'{band}_peak_dB']:.1f} dB above 1/f, location {best[f'{band}_location']:.0%}")
# CHOSEN = {"alpha": 5, "beta": 8}  # manual choice after looking

for band, comp in CHOSEN.items():
    ica.plot_properties(raw, picks=[comp], psd_args=dict(fmax=40))

## 4. Amplitude envelopes of the chosen components

Band-pass each chosen component, then take the Hilbert envelope on the **continuous** signal (no edge artifacts per trial). Components have an arbitrary sign and scale; the envelope removes the sign, and cRQA will use standardized envelopes.

In [ ]:
envelopes = {}
for band, comp in CHOSEN.items():
    x = sources.copy().pick([f"ICA{comp:03d}"])
    x.filter(*BANDS[band], picks="all")  # components are 'misc' channels: picks needed
    x.apply_hilbert(envelope=True, picks="all")
    envelopes[band] = x

fig, ax = plt.subplots(figsize=(10, 3))
t0 = 60
for band, x in envelopes.items():
    sf = x.info["sfreq"]
    seg = x.get_data(start=int(t0 * sf), stop=int((t0 + 10) * sf))[0]
    ax.plot(np.linspace(t0, t0 + 10, seg.size), seg / seg.std(), label=f"{band} (ICA{CHOSEN[band]:03d})")
ax.set(xlabel="time (s)", ylabel="envelope (z)"); ax.legend()

## 5. Save

One file per band, with all events (for cutting trials later), and the choice for the Methods.

In [ ]:
for band, x in envelopes.items():
    x.save(OUT_DIR / f"{SUBJECT}_{band}_envelope_raw.fif", overwrite=True)

choice = scores.set_index("component").loc[list(CHOSEN.values())].round(3)
choice.insert(0, "band", list(CHOSEN))
choice.to_csv(OUT_DIR / f"{SUBJECT}_spatial_filter.csv")
choice

## Questions for Anna

1. One component per band, or several (left and right sensorimotor beta)?
2. Choose components per participant by score, or match them across participants with CORRMAP (an "alpha template"), as for the artifacts?
    Need to look through and see how its turning up.
4. Compare with the simple alternative (average over ROI electrodes) as a check?